# Whakaari Dataset Construction

This notebook builds the Whakaari case-study dataset for Cause–Trigger analysis. It combines waveform-derived hydrothermal/seismic features with weather, gas, and deformation variables on a common hourly grid.

In [ ]:
import sys
from pathlib import Path

import pandas as pd
from obspy.clients.fdsn import Client

PROJECT_ROOT = Path.cwd().parent
SRC_DIR = PROJECT_ROOT / "src/whakaari"
sys.path.append(str(SRC_DIR))

from whakaari_config import (
    WHAKAARI_START,
    WHAKAARI_END,
    WHAKAARI_ERUPTION_TIME,
    TILDE_SUMMARY_URL,
    TILDE_DATA_URL,
    WHAKAARI_LAT,
    WHAKAARI_LON,
    WHAKAARI_WAVEFORM_CONFIG,
    WHAKAARI_EQ_RADIUS_KM,
    WHAKAARI_EQ_MIN_MAGNITUDE,
)

from whakaari_geonet import (
    load_so2_flux,
    load_gnss_deformation,
    load_weather_vars,
    load_local_earthquake_counts,
)

from whakaari_waveform import build_waveform_dataset

from whakaari_dataset import (
    build_master_dataframe,
    prepare_analysis_dataframe,
    scale_analysis_dataframe,
    save_whakaari_datasets,
)

from whakaari_plotting_utils import (
    dataset_health_report,
    plot_with_eruption_time,
    WHAKAARI_ALL_COLS,
)

client = Client("GEONET")

## 1. GeoNet and external data

We retrieve SO₂ flux and GNSS deformation from GeoNet/Tilde, and hourly weather variables from Open-Meteo. API and pressure drop are derived as hydrothermal forcing proxies.

In [ ]:
so2 = load_so2_flux(TILDE_DATA_URL, WHAKAARI_START, WHAKAARI_END)
gnss = load_gnss_deformation(TILDE_DATA_URL, WHAKAARI_START, WHAKAARI_END)
weather_vars = load_weather_vars(
    WHAKAARI_LAT,
    WHAKAARI_LON,
    WHAKAARI_START,
    WHAKAARI_END,
)

In [ ]:
local_eq, local_eq_events = load_local_earthquake_counts(
    client=client,
    start=WHAKAARI_START,
    end=WHAKAARI_END,
    latitude=WHAKAARI_LAT,
    longitude=WHAKAARI_LON,
    radius_km=WHAKAARI_EQ_RADIUS_KM,
    min_magnitude=WHAKAARI_EQ_MIN_MAGNITUDE,
    master_freq="1h",
)

print(local_eq.shape)
display(local_eq.head())
display(local_eq.describe())

print("Event catalogue rows:", len(local_eq_events))
display(local_eq_events.head())

In [ ]:
local_eq["local_eq_count_24h"].plot(
    figsize=(14, 4),
    title="Local earthquake count, rolling 24h"
)

## 2. Waveform feature extraction

Waveform data from WSRZ are processed into hourly seismic features: hydrothermal tremor RMS, spectral ratio, HF event rate, and a continuous tremor-response energy variable.

In [ ]:
WHAKAARI_WAVEFORM_PKL = "../whakaari_data/whakaari_waveform.pkl"

waveform_df, waveform_failures = build_waveform_dataset(
    client=client,
    start=WHAKAARI_START,
    end=WHAKAARI_END,
    cfg=WHAKAARI_WAVEFORM_CONFIG,
    save_path=WHAKAARI_WAVEFORM_PKL,
    overwrite=True, # True will re-download and process waveforms, False will load from pkl
)

## 3. Final hourly dataset

All variables are aligned to a common hourly timeline. Missing values are handled according to variable type, and the final dataset is saved in raw and scaled forms.

In [ ]:
master_df = build_master_dataframe(
    wave=waveform_df,
    weather_vars=weather_vars,
    so2=so2,
    gnss=gnss,
    local_eq=local_eq,
    start=WHAKAARI_START,
    end=WHAKAARI_END,
    master_freq="1h",
)

analysis_df = prepare_analysis_dataframe(master_df)

analysis_scaled, analysis_prepped, scaler = scale_analysis_dataframe(
    analysis_df
)

save_whakaari_datasets(
    master_df=master_df,
    analysis_df=analysis_df,
    analysis_scaled=analysis_scaled,
    output_dir="../whakaari_data",
)

In [ ]:
#load datasets for analysis
#master_df = pd.read_csv("../whakaari_data/whakaari_dataset.csv", parse_dates=["timestamp"]).set_index("timestamp")
#analysis_df = pd.read_csv("../whakaari_data/whakaari_analysis_dataset.csv", parse_dates=["timestamp"]).set_index("timestamp")
#analysis_scaled = pd.read_csv("../whakaari_data/whakaari_analysis_scaled.csv", parse_dates=["timestamp"]).set_index("timestamp")

## 4. Dataset checks

We inspect missingness, summary statistics, and timestamp consistency before using the dataset for causal discovery.

In [ ]:
display(dataset_health_report(master_df, "Whakaari master_df"))
display(dataset_health_report(analysis_df, "Whakaari analysis_df"))
display(dataset_health_report(analysis_scaled, "Whakaari analysis_scaled"))

## 5. Visual inspection

The final variables are plotted with the eruption time marked. This helps assess whether the causal-analysis window and variables are physically meaningful.

In [ ]:
plot_with_eruption_time(
    "../whakaari_data/whakaari_dataset.csv",
    cols=WHAKAARI_ALL_COLS,
    eruption_time=WHAKAARI_ERUPTION_TIME,
    title="Whakaari monitoring variables",
    tick_interval_days=7,
)